# Kafka XGBoost Consumer



In [1]:
import os
# PATH definitions
DATA_DIR = "./dataset"  # đổi nếu dùng Drive
MODEL_DIR = "./models"
TRAIN_PATH = os.path.join(DATA_DIR, "train_data.csv")
VAL_PATH   = os.path.join(DATA_DIR, "val_data.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test_data.csv")

In [2]:
# Columns
TEXT_COL = "Review"
ASPECTS  = ["Price","Shipping","Outlook","Quality","Size","Shop_Service","General","Others"]

# Label mapping
SENT_ID2NAME = {-1: "None", 0: "Negative", 1: "Positive", 2: "Neutral"}
LABEL_VALUES = list(SENT_ID2NAME.keys())  # [-1, 0, 1, 2]
LABEL_NAMES  = list(SENT_ID2NAME.values())  # ["None","Negative","Positive","Neutral"]

In [3]:
# Utilities function 
import re, unicodedata
import pandas as pd

URL_RE = re.compile(r"https?://\S+|www\.\S+")
TAG_RE = re.compile(r"<[^>]+>")
MULTISPACE_RE = re.compile(r"\s+")
VIETNAMESE_BASIC_STOPWORDS = set("""
và hoặc nhưng là thì mà được bị của cho với về từ tới đến nỗi do vì nên nếu khi để bằng như lại đã đang sẽ không chưa chẳng rất quá lắm hơi
này kia nọ đó đây ấy vậy thế sao tại vì do đó tuy nhiên hơn kém chỉ mỗi một các những cái con chiếc đôi đc nhé nha ạ ơi
""".split())

def preprocess_xgb(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = text.strip().lower()
    text = URL_RE.sub(" ", text)
    text = TAG_RE.sub(" ", text)
    text = text.replace("❤️", " yeu ").replace("❤", " yeu ").replace("😍", " yeu ")
    text = re.sub(r"[^\w\sáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ]", " ", text)
    text = unicodedata.normalize("NFC", text)  # normalize accents
    text = MULTISPACE_RE.sub(" ", text).strip()
    tokens = [w for w in text.split() if w not in VIETNAMESE_BASIC_STOPWORDS]
    return " ".join(tokens)

In [4]:
from pyspark.sql import SparkSession
import findspark
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

scala_version = '2.12'
spark_version = '3.5.1'
packages = [ f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}' , 'org.apache.kafka:kafka-clients:3.5.1' ]

findspark.init()
spark = SparkSession.builder.master("local").appName("kafka").config("spark.jars.packages", ",".join(packages)).getOrCreate()


spark.sparkContext.setLogLevel("ERROR")

your 131072x1 screen size is bogus. expect trouble
25/10/18 20:34:44 WARN Utils: Your hostname, DESKTOP-MKOIJLS resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/10/18 20:34:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/d/Study/Cao-hoc/ki-4/big-data/Bai-tap-qua-trinh/bai-6/bai-lam/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/eddiez/.ivy2/cache
The jars for the packages stored in: /home/eddiez/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b3d8e473-d39f-4d6f-bc3b-220becab6e23;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found org.apache.kafka#kafka-clients;3.5.1 in central
	found com.github.luben#zstd-jni;1.5.5-1 in centra

In [5]:
topic_name = 'ABSA'
kafka_server = 'localhost:9092'

df_raw = spark.readStream.format("kafka").option("kafka.bootstrap.servers", kafka_server).option("subscribe", topic_name).load()

In [6]:
# FIXED: Check schema without calling toPandas() on streaming DataFrame
df_raw.printSchema()
print("Schema printed successfully. Note: Cannot call toPandas() on streaming DataFrames.")

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

Schema printed successfully. Note: Cannot call toPandas() on streaming DataFrames.


In [7]:
# FIXED: Remove the problematic loop with toPandas() calls
# Instead, show the structure of the data processing pipeline
print("Kafka streaming DataFrame created successfully.")
print("Schema:", df_raw.columns)
print("Will process data in streaming mode with writeStream.")

Kafka streaming DataFrame created successfully.
Schema: ['key', 'value', 'topic', 'partition', 'offset', 'timestamp', 'timestampType']
Will process data in streaming mode with writeStream.


In [8]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, StructField

# Define input schema for JSON parsing
input_schema = StructType([
    StructField("Review", StringType()),
])

# Parse the JSON data from Kafka
df_input = (
    df_raw
    .selectExpr("CAST(value AS STRING) as json")
    .select(from_json(col("json"), input_schema).alias("data"))
    .select("data.*")
)

print("Data parsing pipeline created successfully.")
df_input.printSchema()

Data parsing pipeline created successfully.
root
 |-- Review: string (nullable = true)



In [9]:
from time import sleep

query = (
    df_input.writeStream
    .format("console")      # print rows to console
    .outputMode("append")   # or "update"
    .start()
)

sleep(5)   # run for 5 seconds
query.stop()


-------------------------------------------
Batch: 0
-------------------------------------------
+------+
|Review|
+------+
+------+



25/10/18 20:34:57 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 1, writer: ConsoleWriter[numRows=20, truncate=true]] is aborting.
25/10/18 20:34:57 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 1, writer: ConsoleWriter[numRows=20, truncate=true]] aborted.


In [10]:
# FIXED: Add proper error handling for model files
import joblib
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType
import numpy as np

# Check if models directory and files exist
if not os.path.exists(MODEL_DIR):
    print(f"Warning: Models directory '{MODEL_DIR}' does not exist.")
    print("Please create the directory and add trained model files:")
    for aspect in ASPECTS:
        print(f"  - {aspect}_xgb.pkl")
        print(f"  - {aspect}_encoder.pkl")
    print("\nFor now, will create dummy predictions.")
    models_exist = False
else:
    # Check if all model files exist
    missing_files = []
    for aspect in ASPECTS:
        model_file = os.path.join(MODEL_DIR, f"{aspect}_xgb.pkl")
        encoder_file = os.path.join(MODEL_DIR, f"{aspect}_encoder.pkl")
        if not os.path.exists(model_file):
            missing_files.append(model_file)
        if not os.path.exists(encoder_file):
            missing_files.append(encoder_file)
    
    if missing_files:
        print(f"Warning: Missing model files: {missing_files}")
        print("Please train and save your models first.")
        print("For now, will create dummy predictions.")
        models_exist = False
    else:
        models_exist = True
        print("All model files found. Loading models...")

25/10/18 20:34:57 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:267)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.$anonfun$run$1(WriteToDataSourceV2Exec.scala:441)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:1397)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.run(WriteToDataSourceV2Exec.scala:486)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.run$(WriteToDataSourceV2Exec.scala:425)
	at org.apache.spark.sql.execution.datasources.v2.DataWritingSparkTask$.run(WriteToDataSourceV2Exec.scala:491)
	at org.apache.spark.sql.execution.datasources.v2.V2TableWriteExec.$anonfun$writeWith

All model files found. Loading models...


In [11]:
# FIXED: Load models with proper error handling
if models_exist:
    # Pre-load models/encoders and broadcast them
    bc_models = {
        aspect: spark.sparkContext.broadcast(joblib.load(os.path.join(MODEL_DIR, f"{aspect}_xgb.pkl")))
        for aspect in ASPECTS
    }
    bc_encoders = {
        aspect: spark.sparkContext.broadcast(joblib.load(os.path.join(MODEL_DIR, f"{aspect}_encoder.pkl")))
        for aspect in ASPECTS
    }
    print("Models loaded and broadcasted successfully.")
else:
    bc_models = {}
    bc_encoders = {}
    print("Using dummy predictions (no actual models loaded).")


# ----------------------
# UDF factory for each aspect
# ----------------------
def make_predict_udf(aspect):
    if models_exist:
        model = bc_models[aspect].value
        encoder = bc_encoders[aspect].value

        @pandas_udf(StringType())
        def predict_udf(texts: pd.Series) -> pd.Series:
            texts_proc = [preprocess_xgb(t) for t in texts]
            preds_encoded = model.predict(texts_proc)
            preds_original = encoder.inverse_transform(preds_encoded)
            return pd.Series([SENT_ID2NAME[int(x)] for x in preds_original])
    else:
        # Dummy UDF when models don't exist
        @pandas_udf(StringType())
        def predict_udf(texts: pd.Series) -> pd.Series:
            # Return random predictions for demonstration
            import random
            return pd.Series([random.choice(["Positive", "Negative", "Neutral", "None"]) for _ in texts])

    return predict_udf

print("UDF factory created successfully.")

/home/eddiez/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/pickle.py:1760: UserWarning: [20:35:05] WARNING: /workspace/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


Models loaded and broadcasted successfully.
UDF factory created successfully.


In [12]:
from pyspark.sql.functions import lit
# Apply UDFs to create prediction columns
df_predictions = df_input.withColumn("student_id", lit("240101030"))

# Move student_id to be the first column
cols = ["student_id"] + [c for c in df_predictions.columns if c != "student_id"]
df_predictions = df_predictions.select(cols)


for aspect in ASPECTS:
    predict_udf = make_predict_udf(aspect)
    df_predictions = df_predictions.withColumn(f"{aspect}_pred", predict_udf("Review"))

# Define columns to show
cols_to_show = ["student_id"] + ["Review"] + [f"{aspect}_pred" for aspect in ASPECTS] 

print("Prediction pipeline created successfully.")
print(f"Columns to display: {cols_to_show}")

Prediction pipeline created successfully.
Columns to display: ['student_id', 'Review', 'Price_pred', 'Shipping_pred', 'Outlook_pred', 'Quality_pred', 'Size_pred', 'Shop_Service_pred', 'General_pred', 'Others_pred']


In [13]:
# Define per-batch function for live console output
def foreach_batch(df, batch_id):
    print(f"\n{'='*50}")
    print(f"Batch {batch_id} - Processing {df.count()} messages")
    print(f"{'='*50}")
    
    # Show the results
    df.select(*cols_to_show).show(5, truncate=False)
    
    # Show some statistics
    print("\nPrediction Summary:")
    for aspect in ASPECTS:
        pred_col = f"{aspect}_pred"
        if pred_col in df.columns:
            df.groupBy(pred_col).count().show()

In [14]:
# FIXED: Start the streaming query properly
print("Starting streaming query...")

# WriteStream triggers every 5 seconds
query = (
    df_predictions.writeStream
    .foreachBatch(foreach_batch)
    .trigger(processingTime="1 seconds")
    .option("checkpointLocation", "/tmp/kafka_checkpoint")  # Add checkpoint for fault tolerance
    .start()
)

print("Stream started successfully!")
print(f"Query status: {query.status}")
print(f"Query id: {query.id}")

Starting streaming query...
Stream started successfully!
Query status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
Query id: b9fdec27-5c7a-4e8d-9546-82b038105564



Batch 68 - Processing 24 messages
+----------+-----------------------------------------------------------------------------------------------------+----------+-------------+------------+------------+---------+-----------------+------------+-----------+
|student_id|Review                                                                                               |Price_pred|Shipping_pred|Outlook_pred|Quality_pred|Size_pred|Shop_Service_pred|General_pred|Others_pred|
+----------+-----------------------------------------------------------------------------------------------------+----------+-------------+------------+------------+---------+-----------------+------------+-----------+
|240101030 |Hàng đẹp nha nhưng là hàng nc ngoài nên giao hơi lâu                                                 |None      |Negative     |Positive    |None        |None     |None             |None        |None       |
|240101030 |Đẹp đúng mẫu mã đa dạng và phong cách của mình để mình bị ạ ở nhà có lạnh    

In [15]:
# Wait for the stream to process some data
# This will run until you interrupt it
try:
    query.awaitTermination(timeout=300)  # Wait for 5 minutes max
except KeyboardInterrupt:
    print("\nStream interrupted by user.")
finally:
    print("Stopping query...")
    query.stop()
    print("Query stopped.")

Stopping query...
Query stopped.


In [ ]:
# Optional: Check query status and metrics
if 'query' in locals() and query.isActive:
    print(f"Query status: {query.status}")
    print(f"Recent progress: {query.recentProgress}")
    print(f"Input rows per second: {query.lastProgress.inputRowsPerSecond}")
else:
    print("Query is not active.")